## tl;dr
冻结76请求的公开CSV复核：76/320请求、836闭合Q槽、685求解=652 strict+33 invalid；全计划3520槽。新增13请求143槽。本notebook只复核公开表，不代替原始数据源链或全部候选误差/CI重算。


## Context & Methods
### Key Assumptions
在notebook与7份CSV同一目录运行，只用Python标准库。输入SHA固定为公开变换后的字节；仅CRLF转LF与研究根前缀相对化。5个代码单元按顺序在普通Python中实际执行，未启动原生Jupyter kernel。失败值保持空，不填0。正式10K15训练与开发5K15物理不合并。

完整私有源链独立数值GO：`f37b9184c587ef5defeabfdd462d1298c651c3a1c862b66f2e354576c270e334`。其3072标量/512指标行/192选择行和整请求bootstrap审核需访问原冻结候选、收据与脚本；这些私有源未包含。本notebook不宣称重复了该审核。


## Data
### 1. 精确公开CSV输入
读取7个相对文件名；核验字节SHA、LF和记录数量。


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv, hashlib, io, json, math

# Run from the directory containing this notebook and these seven public CSVs.
EXPECTED_INPUTS = {'STATUS16.csv': {'sha256': '4e2665e8802cfb71469998557f6d5742708079c702559f37194493a09d8ec1ed', 'bytes': 22064}, 'NEW13_REQUEST_STATUS.csv': {'sha256': 'daa07848866c74d99278ce1f564cac953e1f42caf5afe195e6c282d4635763e7', 'bytes': 1681}, 'NEW13_SELECTED_PHYSICS.csv': {'sha256': '7c653070bfab32b5f5bc05ba7bfbc4571f8419df1478f2d4f937cb918a470fbc', 'bytes': 17030}, 'FREQUENCY_STATUS.csv': {'sha256': '12a29c7d949bc729308055cacec298701da37fe446c849eaf39bb5b3bcca5758', 'bytes': 6613}, 'REQUEST_STATUS.csv': {'sha256': 'cbfeff4af03cac802ee87175f7a6bb26780a23a02bcf05dc5cb6b1ff1de35db0', 'bytes': 104363}, 'METRICS.csv': {'sha256': '40649a14ce9d998716cf0735385a8176b6b9dceb6b28d9b88503dcb8059caf37', 'bytes': 426281}, 'SELECTION_COMPARISON.csv': {'sha256': 'd8d1a92230cf6fceb3a67b5579ac31dcb45e97bee101adaf5d0a3e0d1d6e35d9', 'bytes': 60179}}
tables = {}
for name, expected in EXPECTED_INPUTS.items():
    payload = Path(name).read_bytes()
    assert hashlib.sha256(payload).hexdigest() == expected["sha256"], name
    assert len(payload) == expected["bytes"] and b"\r" not in payload, name
    tables[name] = list(csv.DictReader(io.StringIO(payload.decode("utf-8"))))
counts = {name: len(rows) for name, rows in tables.items()}
assert counts == {"STATUS16.csv":16, "NEW13_REQUEST_STATUS.csv":13,
                  "NEW13_SELECTED_PHYSICS.csv":52, "FREQUENCY_STATUS.csv":16,
                  "REQUEST_STATUS.csv":320, "METRICS.csv":512,
                  "SELECTION_COMPARISON.csv":192}
print(json.dumps({"input_pins": "PASS", "row_counts": counts}, ensure_ascii=False))


{"input_pins": "PASS", "row_counts": {"STATUS16.csv": 16, "NEW13_REQUEST_STATUS.csv": 13, "NEW13_SELECTED_PHYSICS.csv": 52, "FREQUENCY_STATUS.csv": 16, "REQUEST_STATUS.csv": 320, "METRICS.csv": 512, "SELECTION_COMPARISON.csv": 192}}


### 2. 原始分母与完整11门
221全计划解析拒绝不等于76已闭合请求中的46解析拒绝；pending不隐藏。求解工件计数不是IID样本数。


In [2]:
requests = tables["REQUEST_STATUS.csv"]
new_requests = tables["NEW13_REQUEST_STATUS.csv"]
frequency = tables["FREQUENCY_STATUS.csv"]
assert len({r["request_id"] for r in requests}) == 320
assert {int(r["dispatch_order"]) for r in requests} == set(range(320))
closed = [r for r in requests if r["status"] == "ACCOUNTED"]
assert len(closed) == 76 and sum(int(r["N_original"]) for r in requests) == 3520
def total(rows, field):
    return sum(int(r[field]) for r in rows)
expected_closed = {"N_original":836, "N_solved":685, "N_strict_valid":652,
                   "N_invalid":33, "N_analytic_fail":46, "N_gds_fail":73,
                   "N_drc_fail":32, "N_pending":0}
expected_new = {"N_original":143, "N_solved":125, "N_strict_valid":119,
                "N_invalid":6, "N_analytic_fail":0, "N_gds_fail":13,
                "N_drc_fail":5, "N_pending":0}
for rows, expected_counts in ((closed, expected_closed), (new_requests, expected_new)):
    for key, value in expected_counts.items():
        assert total(rows, key) == value, key
for row in requests + new_requests + frequency:
    assert int(row["N_strict_valid"]) + int(row["N_invalid"]) == int(row["N_solved"])
    assert sum(int(row[k]) for k in ("N_solved","N_analytic_fail","N_gds_fail",
               "N_drc_fail","N_emx_not_solved","N_pending")) == int(row["N_original"])
assert total(frequency,"N_accounted_requests") == 76
assert total(frequency,"N_planned_requests") == 320
assert total(frequency,"N_original") == 3520
assert total(frequency,"N_analytic_fail") == 221  # Includes not-yet-accounted requests.
assert total(frequency,"N_pending") == 2509
assert [int(r["frequency_ghz"]) for r in new_requests] == [19,15,5,10,20,6,7,8,9,11,12,13,14]
by_id = {r["request_id"]: r for r in requests}
for r in new_requests:
    assert by_id[r["request_id"]]["status"] == "ACCOUNTED"
    for key in ("frequency_ghz","model_id","dataset_scope","q_proxy"):
        assert r[key] == by_id[r["request_id"]][key]
    other_q = by_id[r["request_id"]]["q_emx"]
    assert (r["q_emx"] == "") == (other_q == "")
    if other_q != "":
        assert float(r["q_emx"]).is_integer() and float(other_q).is_integer()
        assert int(float(r["q_emx"])) == int(float(other_q))
eligible = [r for r in closed if r["q_emx"] != ""]
assert len(eligible) == 9
assert all(int(r["N_strict_valid"]) == int(r["N_original"]) == 11 for r in eligible)
assert all(r["q_emx"] == "" for r in closed if int(r["N_strict_valid"]) != 11)
print(json.dumps({"denominators":"PASS","accounted":76,"planned":320,"slots":3520,
                  "closed_counts":expected_closed,"new_counts":expected_new,"complete11":9}))


{"denominators": "PASS", "accounted": 76, "planned": 320, "slots": 3520, "closed_counts": {"N_original": 836, "N_solved": 685, "N_strict_valid": 652, "N_invalid": 33, "N_analytic_fail": 46, "N_gds_fail": 73, "N_drc_fail": 32, "N_pending": 0}, "new_counts": {"N_original": 143, "N_solved": 125, "N_strict_valid": 119, "N_invalid": 6, "N_analytic_fail": 0, "N_gds_fail": 13, "N_drc_fail": 5, "N_pending": 0}, "complete11": 9}


## Results
### 3. 模型、数据与物理scope
仅核元数据表一致性，不重新加载权重、不重新批准训练或test。


In [3]:
status = tables["STATUS16.csv"]
assert {int(r["frequency_ghz"]) for r in status} == set(range(5,21))
frequency_by_f = {int(r["frequency_ghz"]):r for r in frequency}
mapping = {"physical_accounted_requests":"N_accounted_requests", "physical_original_candidates":"N_original",
           "physical_independent_solves":"N_independent_solves", "physical_strict_valid":"N_strict_valid",
           "physical_strict_invalid":"N_invalid", "physical_pending_candidates":"N_pending"}
for r in status:
    f = int(r["frequency_ghz"]); physical = frequency_by_f[f]
    assert r["training_status"] == "PROVISIONAL_PARTIAL"
    assert int(r["formal_source_geometries"]) == 10000
    assert int(r["formal_train"]) + int(r["formal_validation"]) + int(r["formal_test"]) == int(r["formal_strict"])
    assert r["physical_model_id"] == physical["model_id"]
    assert r["physical_dataset_scope"] == physical["dataset_scope"]
    for target, source in mapping.items(): assert r[target] == physical[source]
    assert (r["physical_dataset_scope"] == "DEVELOPMENT_5K_NOT_FORMAL_10K") == (f == 15)
    assert r["formal_model_fresh_emx"] == ("NOT_RUN" if f == 15 else "PARTIAL_FIXED_REQUEST_EVIDENCE")
assert sum(r["dataset_scope"] == "DEVELOPMENT_5K_NOT_FORMAL_10K" for r in new_requests) == 1
print("16 metadata rows agree; all training PARTIAL, formal10K15 EMX NOT_RUN; development5K15 remains separate.")


16 metadata rows agree; all training PARTIAL, formal10K15 EMX NOT_RUN; development5K15 remains separate.


### 4. 预选候选四指标百分比
使用保存的target、EMX、frozen-grid-proxy复核残差与百分比。该百分比不是联合命中判据；绝对容差仍为[0.125 nH,0.125 nH,1,0.04]。strict-invalid有限descriptor不纳入strict主误差。


In [4]:
features = ("Lp_nH","Ls_nH","Qmin","K_abs")  # Existing CSV names: Qmin=min(Qp,Qs).
selected = tables["NEW13_SELECTED_PHYSICS.csv"]
selected_groups = defaultdict(list)
for r in selected: selected_groups[r["request_id"]].append(r)
assert set(selected_groups) == {r["request_id"] for r in new_requests}
numeric_fields = ("actual_emx","emx_minus_target","emx_minus_frozen_proxy",
                  "target_absolute_relative_percent","proxy_absolute_relative_percent")
checked_percentage_values = 0
missing_rows = 0
invalid_rows = 0
for request in new_requests:
    group = selected_groups[request["request_id"]]
    assert len(group) == 4 and {r["feature"] for r in group} == set(features)
    for row in group:
        assert row["candidate_id"] == request["request_id"] + "-q" + request["q_proxy"]
        for key in ("frequency_ghz","model_id","dataset_scope","q_proxy"):
            assert row[key] == request[key]
        assert (row["q_emx"] == "") == (request["q_emx"] == "")
        if row["q_emx"] != "":
            assert float(row["q_emx"]).is_integer() and float(request["q_emx"]).is_integer()
            assert int(float(row["q_emx"])) == int(float(request["q_emx"]))
        if row["stage"] == "GDS_FAIL":
            assert all(row[key] == "" for key in numeric_fields)
            assert row["strict_valid"] == row["strict_joint_hit"] == ""
            missing_rows += 1
            continue
        assert row["stage"] in ("SOLVED_STRICT_VALID","SOLVED_INVALID")
        target, actual, proxy = (float(row[k]) for k in ("target","actual_emx","frozen_grid_proxy"))
        assert all(math.isfinite(v) for v in (target,actual,proxy))
        assert math.isclose(actual-target,float(row["emx_minus_target"]),rel_tol=1e-10,abs_tol=1e-12)
        assert math.isclose(actual-proxy,float(row["emx_minus_frozen_proxy"]),rel_tol=1e-10,abs_tol=1e-12)
        for reference,key in ((target,"target_absolute_relative_percent"),(proxy,"proxy_absolute_relative_percent")):
            # These selected rows are nonzero; do not manufacture a near-zero percentage.
            assert abs(reference) > 1e-12
            expected_percent = abs(actual-reference) / abs(reference) * 100
            assert math.isclose(expected_percent,float(row[key]),rel_tol=1e-10,abs_tol=1e-10)
            checked_percentage_values += 1
        if row["stage"] == "SOLVED_INVALID":
            assert row["strict_valid"] == "False" and row["strict_joint_hit"] == "False"
            invalid_rows += 1
        else: assert row["strict_valid"] == "True"
assert missing_rows == 8 and invalid_rows == 4 and checked_percentage_values == 88
print(json.dumps({"selected_four_feature_rows":52,"missing_GDS_rows":8,"invalid_descriptor_rows":4,
                  "saved_target_and_proxy_percent_values_checked":checked_percentage_values,
                  "invalid_values_excluded_from_strict_primary":True}))


{"selected_four_feature_rows": 52, "missing_GDS_rows": 8, "invalid_descriptor_rows": 4, "saved_target_and_proxy_percent_values_checked": 88, "invalid_values_excluded_from_strict_primary": true}


### 5. 已保存统计表结构
只检查六误差字段、范围和分组齐全；不从不足的公开selected行伪重算all-candidate MAE/P95/CI。


In [5]:
metrics = tables["METRICS.csv"]
selection = tables["SELECTION_COMPARISON.csv"]
required_metrics = {"frequency_ghz","model_id","dataset_scope","estimand","validity","comparison",
                    "feature","unit","n","mae","rmse","bias","abs_error_p50","abs_error_p90",
                    "abs_error_p95","ci_status","n_request_groups","N_original","N_solved",
                    "N_strict_valid","N_invalid","N_accounted_candidates"}
assert required_metrics <= set(metrics[0])
assert len(metrics) == 512 and len(selection) == 192
assert {r["feature"] for r in metrics} == set(features)
assert {r["estimand"] for r in metrics} == {"all_candidates","selected_q_proxy"}
assert {r["comparison"] for r in metrics} == {"emx_minus_target","emx_minus_frozen_proxy"}
assert len({(r["frequency_ghz"],r["model_id"],r["dataset_scope"],r["estimand"],
             r["validity"],r["comparison"],r["feature"]) for r in metrics}) == 512
for row in metrics + selection:
    assert (row["dataset_scope"] == "DEVELOPMENT_5K_NOT_FORMAL_10K") == (int(row["frequency_ghz"]) == 15)
    for key in ("mae","rmse","bias","abs_error_p50","abs_error_p90","abs_error_p95"):
        assert row[key] == "" or math.isfinite(float(row[key]))
assert {r["strategy"] for r in selection} == {"fixed_q15","q_proxy","q_emx"}
print("PASS: six error fields and distinct estimands/scopes are present. Raw-candidate MAE/CI were NOT recomputed.")
print("Full private-chain numerical QA: f37b9184c587ef5defeabfdd462d1298c651c3a1c862b66f2e354576c270e334")
print("This companion checks published CSV consistency only, not raw S4P identity or population accuracy.")


PASS: six error fields and distinct estimands/scopes are present. Raw-candidate MAE/CI were NOT recomputed.
Full private-chain numerical QA: f37b9184c587ef5defeabfdd462d1298c651c3a1c862b66f2e354576c270e334
This companion checks published CSV consistency only, not raw S4P identity or population accuracy.


## Takeaways
公开CSV分母与结构检查、52个预选四特征行及88个已保存target/proxy相对百分比复核通过。5GHz Q10与8GHz Q11预选GDS_FAIL保留空值；20GHz Q16预选strict-invalid，不因Q百分比小宣称合格。9个完整strict11请求才支持q_emx共同集。全部模型仍PARTIAL/PROVISIONAL，有限请求不代表随机总体准确率或跨频因果冠军。图的视觉GO另见公开包收据。
